# Extended EOF Analysis

---

## Overview


## Prerequisites

| Concepts | Importance | Notes |
| --- | --- | --- |
|  |  | |

- **Time to learn**: -- minutes.

---

## Imports

In [ ]:
import s3fs
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import CenteredNorm
import xarray as xr
from scipy import linalg as la
import cartopy.crs as ccrs

## Access data

In [ ]:
URL = 'https://js2.jetstream-cloud.org:8001/' #Locate and read a file
fs = s3fs.S3FileSystem(anon=True, client_kwargs=dict(endpoint_url=URL))

olr_noaa_store = s3fs.S3Map(
    root=f'pythia/olr_noaa.zarr',
    s3=fs,
    check=False
)

In [ ]:
olr = xr.open_zarr(olr_noaa_store)
olr = olr.rename_vars({'__xarray_dataarray_variable__': 'olr'}).sel(time=slice('1987', None))
olr = olr.resample(time='ME').mean()

In [ ]:
olr.olr.isel(time=0).plot()

## Deseasonalize

In [ ]:
# olr_gb = olr.olr.groupby('time.day')
# olr_anom = olr_gb - olr_gb.mean(dim='time')

In [ ]:
n_time, n_lat, n_lon = olr.olr.shape
data_2d = olr.olr.values.reshape(n_time, -1)

n_harmonics=4
year_period=12

# Build design matrix: 1 constant + n_harmonics sin/cos pairs
t = np.arange(n_time)
X = np.ones((n_time, 2 * n_harmonics + 1))
for i in range(1, n_harmonics + 1):
    X[:, 2*i - 1] = np.sin(i * 2 * np.pi * t / year_period)
    X[:, 2*i] = np.cos(i * 2 * np.pi * t / year_period)

# Solve via least squares and subtract seasonal component
coeffs = np.linalg.lstsq(X, data_2d, rcond=None)[0]
olr_anom = data_2d - X @ coeffs

In [ ]:
plt.imshow(olr_anom)

## Construct lagged matrix

In [ ]:
data_matrix = olr_anom

n_time, n_space = data_matrix.shape

max_lag = 3
lagged_matrix = np.zeros((n_time - 6*max_lag, n_space * (max_lag + 1))) * np.nan

for i in range(max_lag + 1):
    if (max_lag - i) != 0:
        lagged_matrix[:, i*n_space: i*n_space + n_space] = data_matrix[6*i:6*i - 6*max_lag, :]
    else:
        lagged_matrix[:, i*n_space: i*n_space + n_space] = data_matrix[6*i:, :]

In [ ]:
lagged_matrix.shape

## Extended EOF analysis

In [ ]:
data_3d = olr.olr

nan_indices = np.where(np.isnan(lagged_matrix[0]))
lagged_matrix_no_nan = np.delete(lagged_matrix, nan_indices, 1)

cov_matrix = np.dot(lagged_matrix_no_nan, lagged_matrix_no_nan.T)
eigenvalues, eigenvectors = la.eig(cov_matrix)

total_variance = np.sum(eigenvalues)
variance_explained = (eigenvalues / total_variance) * 100

# Project eigenvectors onto the data to obtain the EOF spatial patterns
eof_modes = np.dot(eigenvectors.T, lagged_matrix_no_nan)

eof_with_nan = np.copy(lagged_matrix) * np.nan
all_space_indices = np.arange(lagged_matrix.shape[1])
valid_indices = np.setdiff1d(all_space_indices, nan_indices)
eof_with_nan[:, valid_indices] = eof_modes
eof_modes = eof_with_nan

n_modes, n_space_total = eof_modes.shape
eeofs = [np.copy(data_3d) * np.nan] * (max_lag + 1)

# Split the EOF matrix into its lag blocks
n_space_per_lag = int(n_space_total / (max_lag + 1))
eof_split = np.zeros((max_lag + 1, n_modes, n_space_per_lag)) * np.nan

for i in range(max_lag + 1):
    eof_split[i, :, :] = eof_modes[:, i*n_space_per_lag: (i + 1)*n_space_per_lag]

for i in range(max_lag + 1):
    eeofs[i] = eof_split[i, :, :].reshape(
        data_3d.shape[0] - 6*max_lag, data_3d.shape[1], data_3d.shape[2])

eeofs = np.array(eeofs)

In [ ]:
plt.imshow(eeofs[0, 0])

In [ ]:
n_lag, n_modes, _, _ = eeofs.shape
eeof_coords = {
    'lag': np.arange(max_lag + 1),
    'mode': np.arange(n_modes),
    'lat': olr.lat.values,
    'lon': olr.lon.values,
}
eeof_da = xr.DataArray(
    eeofs.astype('float32'),
    coords=eeof_coords,
    name='eof',
    attrs={'standard_name': 'EEOF'},
)
eeof_da

In [ ]:
pc_coords = {
    'mode': np.arange(n_time - 6*max_lag),
    'time': olr.time.values[0:-6*max_lag],
}
pcs_da = xr.DataArray(
    eigenvectors.T.astype('float32'),
    coords=pc_coords,
    name='pc',
    attrs={'standard_name': 'Principal Components / Eigenvectors'},
)

In [ ]:
variance_explained_coords = {
    'mode': np.arange(len(variance_explained)),
}
variance_explained_da = xr.DataArray(
    variance_explained.astype('float32'),
    coords=variance_explained_coords,
    name='var_exp',
    attrs={'standard_name': 'Variance Explained [%]'},
)

## Plot results

In [ ]:
variance_explained_da[:10].plot.scatter()

The leading mode is by far the most important.

In [ ]:
plots = eeof_da.isel(mode=slice(0, 4)).plot.contourf(levels=21, col='mode', row='lag', subplot_kws={'projection': ccrs.PlateCarree()}, figsize=(15, 5))

[ax.coastlines() for ax in plots.axs.flatten()];

In [ ]:
pc_plots = pcs_da.isel(mode=slice(0, 4)).plot(row='mode', figsize=(10, 5))
[ax.grid() for ax in pc_plots.axs.flatten()];

A closer look at EEOF2 at different lags:

In [ ]:
fig = plt.figure(figsize=(10, 5), layout='constrained')
ax = plt.subplot(111, projection=ccrs.PlateCarree(central_longitude=180))
plot = ax.contourf(eeof_da.lon, eeof_da.lat, eeof_da.isel(mode=1, lag=0), levels=21, transform=ccrs.PlateCarree(), cmap='RdBu_r', norm=CenteredNorm())
ax.coastlines()
ax.gridlines(draw_labels=True, alpha=0.5)
plt.colorbar(plot, orientation='horizontal')
plt.title('Mode 2, lag 0 months')

In [ ]:
fig = plt.figure(figsize=(10, 5), layout='constrained')
ax = plt.subplot(111, projection=ccrs.PlateCarree(central_longitude=180))
plot = ax.contourf(eeof_da.lon, eeof_da.lat, eeof_da.isel(mode=1, lag=1), levels=21, transform=ccrs.PlateCarree(), cmap='RdBu_r', norm=CenteredNorm())
ax.coastlines()
ax.gridlines(draw_labels=True, alpha=0.5)
plt.colorbar(plot, orientation='horizontal')
plt.title('Mode 2, lag 6 months')

In [ ]:
fig = plt.figure(figsize=(10, 5), layout='constrained')
ax = plt.subplot(111, projection=ccrs.PlateCarree(central_longitude=180))
plot = ax.contourf(eeof_da.lon, eeof_da.lat, eeof_da.isel(mode=1, lag=2), levels=21, transform=ccrs.PlateCarree(), cmap='RdBu_r', norm=CenteredNorm())
ax.coastlines()
ax.gridlines(draw_labels=True, alpha=0.5)
plt.colorbar(plot, orientation='horizontal')
plt.title('Mode 2, lag 12 months')

In [ ]:
fig = plt.figure(figsize=(10, 5), layout='constrained')
ax = plt.subplot(111, projection=ccrs.PlateCarree(central_longitude=180))
plot = ax.contourf(eeof_da.lon, eeof_da.lat, eeof_da.isel(mode=1, lag=3), levels=21, transform=ccrs.PlateCarree(), cmap='RdBu_r', norm=CenteredNorm())
ax.coastlines()
ax.gridlines(draw_labels=True, alpha=0.5)
plt.colorbar(plot, orientation='horizontal')
plt.title('Mode 2, lag 18 months')

The OLR anomalies change sign over ~1 year, and it seems to be related to ENSO, since there are peaks in the first two PCs just before 1998 and 2016, years with strong positive ENSO.

---

## Summary

### What's next?


## Resources and references